# Slopodar Voice-Distance Calibration Analysis

> **Analyst:** Python/scipy reproduction of the JS calibration (originally by AnotherPair)  
> **Data:** `calibration-data-v3.tsv` — 47 entries, 6 categories, 29 features  
> **Extension:** slopodar v0.3.0 (`content.js`)  
> **Date:** 2026-03-01  

---

## Scope and Limitations

This notebook reproduces and extends the voice-distance calibration analysis for the [slopodar](https://oceanheart.ai/slopodar/) Chrome extension.
The slopodar measures text features — contractions, first-person pronouns, transition words, nominalisation density, rhetorical structures — to estimate how far a piece of writing sits from a human voice baseline.

**What this is:** Exploratory data analysis on a small, hand-curated calibration dataset. The statistics are real; the generalisability is narrow.

**What this is not:** A classifier you should trust in production. n=47 across 6 categories. The calibration set is dominated by English-language tech blog writers (mostly male, mostly American, mostly writing about software). The features were designed by a single researcher. The ground truth labels are editorial judgments, not verified provenance. See slopodar entries [#13 Construct Drift](https://oceanheart.ai/slopodar/construct-drift/), [#14 Demographic Bake-In](https://oceanheart.ai/slopodar/demographic-bake-in/), and [#15 Monoculture Analysis](https://oceanheart.ai/slopodar/monoculture-analysis/) for the known confounds.

**Demographic disclosure:** The calibration corpus is English tech blogs, 2000-2024. Category A (pre-LLM human) is 19 essays by 9 writers, all male, all writing about technology. Category C (AI company) is 7 corporate blog posts from 3 companies. These demographics limit what the analysis can claim about "human writing" vs "AI writing" in general. It can only claim things about *this specific population of writers and this specific population of AI-generated corporate content*.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from pathlib import Path
import re
import glob
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

# Tokyo Night-inspired palette
TOKYO_BG = '#1a1b26'
TOKYO_FG = '#c0caf5'
TOKYO_BLUE = '#7aa2f7'
TOKYO_GREEN = '#9ece6a'
TOKYO_RED = '#f7768e'
TOKYO_YELLOW = '#e0af68'
TOKYO_PURPLE = '#bb9af7'
TOKYO_CYAN = '#7dcfff'
TOKYO_ORANGE = '#ff9e64'
TOKYO_COMMENT = '#565f89'
TOKYO_GRID = '#292e42'

CAT_COLORS = {
    'A-human-pre': TOKYO_GREEN,
    'B-human-post': TOKYO_CYAN,
    'C-ai-co': TOKYO_RED,
    'D-suspected': TOKYO_ORANGE,
    'E-willison': TOKYO_PURPLE,
    'F-captain': TOKYO_YELLOW,
}

CAT_LABELS = {
    'A-human-pre': 'A: Human (pre-LLM)',
    'B-human-post': 'B: Human (post-LLM)',
    'C-ai-co': 'C: AI company blogs',
    'D-suspected': 'D: Suspected LLM',
    'E-willison': 'E: Willison',
    'F-captain': 'F: Captain',
}

# Apply dark theme globally
plt.rcParams.update({
    'figure.facecolor': TOKYO_BG,
    'axes.facecolor': TOKYO_BG,
    'axes.edgecolor': TOKYO_COMMENT,
    'axes.labelcolor': TOKYO_FG,
    'text.color': TOKYO_FG,
    'xtick.color': TOKYO_FG,
    'ytick.color': TOKYO_FG,
    'grid.color': TOKYO_GRID,
    'legend.facecolor': TOKYO_BG,
    'legend.edgecolor': TOKYO_COMMENT,
    'legend.labelcolor': TOKYO_FG,
    'figure.dpi': 120,
    'font.size': 10,
    'axes.titlesize': 12,
    'axes.labelsize': 10,
})

print('Libraries loaded. Tokyo Night theme applied.')

---
## Section 1: Data Loading & Exploration

In [ ]:
# Load TSV calibration data
tsv_path = Path('../slopodar-ext/calibration-data-v3.tsv')
df = pd.read_csv(tsv_path, sep='\t')

print(f'Loaded {len(df)} entries from {tsv_path.name}')
print(f'Columns ({len(df.columns)}): {list(df.columns)}')
print()
print('Category counts:')
print(df['cat'].value_counts().sort_index())
print()
print(f'Word count range: {df["totalWords"].min()} - {df["totalWords"].max()}')
print(f'Mean words per entry: {df["totalWords"].mean():.0f}')

In [ ]:
# Summary statistics by category for key voice metrics
voice_metrics = ['contractionPer1k', 'firstPersonPer1k', 'questionRate',
                 'transitionPer1k', 'nomDensity']

print('=== Voice Metric Summary by Category ===')
for metric in voice_metrics:
    print(f'\n--- {metric} ---')
    summary = df.groupby('cat')[metric].agg(['count', 'mean', 'std', 'min', 'max'])
    summary = summary.round(2)
    print(summary.to_string())

In [ ]:
# Distribution plots: box + strip for each voice metric, colored by category
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Voice Metric Distributions by Category', fontsize=14, fontweight='bold')

all_metrics = voice_metrics + ['emdashPer1k']
metric_labels = {
    'contractionPer1k': 'Contractions per 1k words',
    'firstPersonPer1k': 'First-person per 1k words',
    'questionRate': 'Question rate (%)',
    'transitionPer1k': 'Transitions per 1k words',
    'nomDensity': 'Nominalisation density (%)',
    'emdashPer1k': 'Em-dashes per 1k words',
}

cat_order = ['A-human-pre', 'B-human-post', 'C-ai-co', 'D-suspected', 'E-willison', 'F-captain']
palette = [CAT_COLORS[c] for c in cat_order if c in df['cat'].values]
cats_present = [c for c in cat_order if c in df['cat'].values]

for idx, metric in enumerate(all_metrics):
    ax = axes[idx // 3][idx % 3]
    sns.boxplot(data=df, x='cat', y=metric, order=cats_present,
                palette=palette, ax=ax, linewidth=0.8, fliersize=0,
                boxprops=dict(alpha=0.4))
    sns.stripplot(data=df, x='cat', y=metric, order=cats_present,
                  palette=palette, ax=ax, size=5, alpha=0.8, jitter=0.15)
    ax.set_title(metric_labels.get(metric, metric), fontsize=10)
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.tick_params(axis='x', rotation=45, labelsize=7)
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Distribution plots for rhetoric metrics
rhetoric_metrics = ['epigramPer1k', 'isocolonPer1k', 'antithesisPer1k', 'anadipPer1k']
rhetoric_labels = {
    'epigramPer1k': 'Epigrams per 1k words',
    'isocolonPer1k': 'Isocolon per 1k words',
    'antithesisPer1k': 'Antithesis per 1k words',
    'anadipPer1k': 'Anadiplosis per 1k words',
}

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.suptitle('Structural Rhetoric Metrics by Category', fontsize=14, fontweight='bold')

for idx, metric in enumerate(rhetoric_metrics):
    ax = axes[idx]
    sns.boxplot(data=df, x='cat', y=metric, order=cats_present,
                palette=palette, ax=ax, linewidth=0.8, fliersize=0,
                boxprops=dict(alpha=0.4))
    sns.stripplot(data=df, x='cat', y=metric, order=cats_present,
                  palette=palette, ax=ax, size=5, alpha=0.8, jitter=0.15)
    ax.set_title(rhetoric_labels.get(metric, metric), fontsize=10)
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.tick_params(axis='x', rotation=45, labelsize=7)
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

---
## Section 2: Reproduce JS Calibration Findings

The JS analysis (`calibration-results.md`) reported these top-5 discriminators between Category A (pre-LLM human) and Category C (AI company blogs), ranked by Cohen's d magnitude:

| Rank | Feature | d (JS) | Direction |
|------|---------|--------|-----------|
| 1 | Transition density | -4.36 | AI higher |
| 2 | Nom density | -2.92 | AI higher |
| 3 | Question rate | 1.78 | Human higher |
| 4 | First-person rate | 1.28 | Human higher |
| 5 | Contraction rate | 0.96 | Human higher |

Let's reproduce these with the v3 data (which now includes more C, D, E, and F samples).

In [ ]:
def cohens_d(group1, group2):
    """Cohen's d effect size. Positive = group1 > group2."""
    n1, n2 = len(group1), len(group2)
    if n1 < 2 or n2 < 2:
        return np.nan
    mean1, mean2 = group1.mean(), group2.mean()
    var1, var2 = group1.var(ddof=1), group2.var(ddof=1)
    # pooled std
    pooled_std = np.sqrt(((n1 - 1) * var1 + (n2 - 1) * var2) / (n1 + n2 - 2))
    if pooled_std == 0:
        return np.inf if mean1 != mean2 else 0.0
    return (mean1 - mean2) / pooled_std

# Compare A vs C across all numeric features
cat_a = df[df['cat'] == 'A-human-pre']
cat_c = df[df['cat'] == 'C-ai-co']

print(f'Category A: n={len(cat_a)}, Category C: n={len(cat_c)}')
print()

# Compute effect sizes for all numeric columns
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
effect_sizes = []

for col in numeric_cols:
    d = cohens_d(cat_a[col], cat_c[col])
    effect_sizes.append({
        'feature': col,
        'cohens_d': d,
        'abs_d': abs(d) if not np.isinf(d) else 999,
        'A_mean': cat_a[col].mean(),
        'C_mean': cat_c[col].mean(),
        'direction': 'Human higher' if d > 0 else 'AI higher',
    })

es_df = pd.DataFrame(effect_sizes).sort_values('abs_d', ascending=False)

print('=== Cohen\'s d: Category A (human) vs Category C (AI) ===')
print('Ranked by |d| descending. Positive d = humans score higher.\n')
for i, row in es_df.head(15).iterrows():
    d_str = f'{row["cohens_d"]:+.2f}' if not np.isinf(row['cohens_d']) else 'inf'
    print(f'  {row["feature"]:25s}  d = {d_str:>8s}  A={row["A_mean"]:7.2f}  C={row["C_mean"]:7.2f}  {row["direction"]}')

In [ ]:
# Highlight the top 5 voice discriminators and compare to JS findings
js_top5 = {
    'transitionPer1k': -4.36,
    'nomDensity': -2.92,
    'questionRate': 1.78,
    'firstPersonPer1k': 1.28,
    'contractionPer1k': 0.96,
}

print('=== JS vs Python Effect Size Comparison ===')
print(f'{"Feature":25s} {"JS d":>8s} {"Python d":>10s} {"Delta":>8s}')
print('-' * 55)
for feat, js_d in js_top5.items():
    py_row = es_df[es_df['feature'] == feat].iloc[0]
    py_d = py_row['cohens_d']
    delta = py_d - js_d if not np.isinf(py_d) else float('nan')
    py_str = f'{py_d:+.2f}' if not np.isinf(py_d) else 'inf'
    delta_str = f'{delta:+.2f}' if not np.isnan(delta) else 'n/a'
    print(f'{feat:25s} {js_d:>+8.2f} {py_str:>10s} {delta_str:>8s}')

print()
print('Note: JS analysis used v2 data (29 entries, 3 C samples).')
print(f'Python analysis uses v3 data ({len(df)} entries, {len(cat_c)} C samples).')
print('Differences are expected due to expanded dataset.')

---
## Section 3: Formal Statistical Tests

The JS analysis computed effect sizes but couldn't do formal hypothesis testing. With scipy we can.

**Approach:**
- Mann-Whitney U (non-parametric, appropriate for small n, no normality assumption)
- Kruskal-Wallis (non-parametric one-way ANOVA across all 6 categories)
- Bonferroni correction for multiple comparisons
- Bootstrap 95% CIs for effect sizes

In [ ]:
# Mann-Whitney U: A vs C for each voice metric
voice_cols = ['contractionPer1k', 'firstPersonPer1k', 'questionRate',
              'transitionPer1k', 'nomDensity', 'hedgePer1k',
              'emdashPer1k', 'epigramPer1k', 'isocolonPer1k',
              'antithesisPer1k', 'anadipPer1k']

n_tests = len(voice_cols)
alpha = 0.05
bonferroni_alpha = alpha / n_tests

print(f'=== Mann-Whitney U Tests: A (human, n={len(cat_a)}) vs C (AI, n={len(cat_c)}) ===')
print(f'Bonferroni-corrected alpha = {alpha}/{n_tests} = {bonferroni_alpha:.4f}\n')
print(f'{"Feature":25s} {"U":>8s} {"p-value":>12s} {"p (Bonf)":>12s} {"Sig?":>6s} {"d":>8s}')
print('-' * 75)

mw_results = []
for col in voice_cols:
    a_vals = cat_a[col].dropna()
    c_vals = cat_c[col].dropna()
    if len(a_vals) < 2 or len(c_vals) < 2:
        continue
    u_stat, p_val = stats.mannwhitneyu(a_vals, c_vals, alternative='two-sided')
    p_bonf = min(p_val * n_tests, 1.0)
    d = cohens_d(a_vals, c_vals)
    sig = '***' if p_bonf < 0.001 else ('**' if p_bonf < 0.01 else ('*' if p_bonf < 0.05 else 'ns'))
    d_str = f'{d:+.2f}' if not np.isinf(d) else 'inf'
    print(f'{col:25s} {u_stat:>8.1f} {p_val:>12.6f} {p_bonf:>12.6f} {sig:>6s} {d_str:>8s}')
    mw_results.append({
        'feature': col, 'U': u_stat, 'p': p_val,
        'p_bonferroni': p_bonf, 'significant': p_bonf < alpha,
        'cohens_d': d
    })

mw_df = pd.DataFrame(mw_results).sort_values('p')
print(f'\nSignificant after Bonferroni correction: {mw_df["significant"].sum()} / {len(mw_df)}')

In [ ]:
# Kruskal-Wallis: across ALL categories for each metric
print('=== Kruskal-Wallis H-Tests: Across All Categories ===')
print(f'Tests whether distributions differ across any category pair.\n')
print(f'{"Feature":25s} {"H":>8s} {"p-value":>12s} {"p (Bonf)":>12s} {"Sig?":>6s}')
print('-' * 65)

for col in voice_cols:
    groups = [grp[col].dropna().values for _, grp in df.groupby('cat') if len(grp[col].dropna()) >= 2]
    if len(groups) < 2:
        continue
    h_stat, p_val = stats.kruskal(*groups)
    p_bonf = min(p_val * n_tests, 1.0)
    sig = '***' if p_bonf < 0.001 else ('**' if p_bonf < 0.01 else ('*' if p_bonf < 0.05 else 'ns'))
    print(f'{col:25s} {h_stat:>8.2f} {p_val:>12.6f} {p_bonf:>12.6f} {sig:>6s}')

In [ ]:
# Bootstrap 95% CIs for Cohen's d (A vs C)
rng = np.random.default_rng(42)
n_bootstrap = 10000

print('=== Bootstrap 95% CIs for Cohen\'s d (A vs C) ===')
print(f'{n_bootstrap} resamples per metric\n')
print(f'{"Feature":25s} {"d":>8s} {"95% CI lower":>14s} {"95% CI upper":>14s}')
print('-' * 65)

bootstrap_results = []
for col in voice_cols:
    a_vals = cat_a[col].dropna().values
    c_vals = cat_c[col].dropna().values
    if len(a_vals) < 2 or len(c_vals) < 2:
        continue
    
    boot_ds = []
    for _ in range(n_bootstrap):
        a_boot = rng.choice(a_vals, size=len(a_vals), replace=True)
        c_boot = rng.choice(c_vals, size=len(c_vals), replace=True)
        boot_d = cohens_d(pd.Series(a_boot), pd.Series(c_boot))
        if not np.isinf(boot_d) and not np.isnan(boot_d):
            boot_ds.append(boot_d)
    
    if len(boot_ds) > 100:
        ci_low = np.percentile(boot_ds, 2.5)
        ci_high = np.percentile(boot_ds, 97.5)
        d = cohens_d(pd.Series(a_vals), pd.Series(c_vals))
        d_str = f'{d:+.2f}' if not np.isinf(d) else 'inf'
        print(f'{col:25s} {d_str:>8s} {ci_low:>+14.2f} {ci_high:>+14.2f}')
        bootstrap_results.append({
            'feature': col, 'd': d, 'ci_low': ci_low, 'ci_high': ci_high,
            'valid_boots': len(boot_ds)
        })

In [ ]:
# Forest plot of effect sizes with bootstrap CIs
boot_df = pd.DataFrame(bootstrap_results).sort_values('d', key=abs, ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
y_pos = range(len(boot_df))

colors = [TOKYO_RED if d < 0 else TOKYO_GREEN for d in boot_df['d']]
ax.barh(y_pos, boot_df['d'], color=colors, alpha=0.6, height=0.6)
ax.errorbar(boot_df['d'], y_pos,
            xerr=[boot_df['d'] - boot_df['ci_low'], boot_df['ci_high'] - boot_df['d']],
            fmt='none', ecolor=TOKYO_FG, elinewidth=1, capsize=3)

ax.set_yticks(y_pos)
ax.set_yticklabels(boot_df['feature'])
ax.axvline(x=0, color=TOKYO_COMMENT, linestyle='--', linewidth=0.8)
ax.axvline(x=-0.8, color=TOKYO_RED, linestyle=':', linewidth=0.5, alpha=0.5)
ax.axvline(x=0.8, color=TOKYO_GREEN, linestyle=':', linewidth=0.5, alpha=0.5)
ax.set_xlabel("Cohen's d (positive = human higher)")
ax.set_title('Effect Sizes: Human (A) vs AI (C) with 95% Bootstrap CIs')
ax.grid(axis='x', alpha=0.3)

# Annotate regions
ax.text(0.8, len(boot_df) - 0.5, '|d|>0.8 = large', fontsize=7, color=TOKYO_COMMENT, ha='left')

plt.tight_layout()
plt.show()

---
## Section 4: Score Oceanheart Pages

Apply the exact same heuristics as the slopodar extension (v0.3 `content.js`) to all markdown files in `sites/oceanheart/content/`.

The analysis engine is ported precisely: same regex patterns, same sentence splitting, same feature definitions.

In [ ]:
# ── Slopodar analysis engine, ported from content.js v0.3 ──

# Exact regex patterns from content.js

# Contractions (line 194 of content.js)
CONTRACT_RE = re.compile(
    r"\b(don't|won't|can't|it's|I'm|you're|they're|we're|isn't|aren't|"
    r"wasn't|weren't|hasn't|haven't|hadn't|couldn't|wouldn't|shouldn't|"
    r"didn't|doesn't|I'll|you'll|we'll|they'll|I've|you've|we've|they've|"
    r"I'd|you'd|we'd|they'd|he's|she's|that's|what's|there's|here's|let's|who's)\b",
    re.IGNORECASE
)

# First-person pronouns (line 199 of content.js)
FIRST_PERSON_RE = re.compile(r'\b(I|me|my|mine|myself)\b')

# Nominalisation / abstract nouns (line 186 of content.js)
NOM_RE = re.compile(r'\b\w+(?:tion|ment|ness|ity|ence|ance)\b', re.IGNORECASE)

# Transition words (line 272 of content.js)
TRANSITION_RE = re.compile(
    r'\b(However|Moreover|Furthermore|Additionally|Consequently|Nevertheless|'
    r'Nonetheless|Importantly|Specifically|Ultimately|Fundamentally|Indeed|'
    r'Notably|Interestingly|Crucially|Essentially|Particularly)\b'
)

# Antithesis patterns (lines 226-233 of content.js)
ANTI_PATTERNS = [
    re.compile(r'\bnot\s+\w+[,;]\s*but\b', re.IGNORECASE),
    re.compile(r'\b\w+[,;]\s*not\s+\w+', re.IGNORECASE),
    re.compile(r'\brather\s+than\b', re.IGNORECASE),
    re.compile(r'\binstead\s+of\b', re.IGNORECASE),
    re.compile(r'\bnothing\s+was\b', re.IGNORECASE),
    re.compile(r'\bnot\s+through\b.*\bthrough\b', re.IGNORECASE),
    re.compile(r'\bnot\s+just\b', re.IGNORECASE),
]


def strip_hugo_frontmatter(text):
    """Remove Hugo TOML frontmatter (between +++ delimiters)."""
    return re.sub(r'^\+\+\+.*?\+\+\+\s*', '', text, count=1, flags=re.DOTALL)


def strip_markdown(text):
    """Strip markdown syntax: code blocks, headers, links, images, blockquotes."""
    # Fenced code blocks
    text = re.sub(r'```[\s\S]*?```', '', text)
    # Inline code
    text = re.sub(r'`[^`]+`', '', text)
    # Images
    text = re.sub(r'!\[.*?\]\(.*?\)', '', text)
    # Links — keep link text
    text = re.sub(r'\[([^\]]+)\]\([^)]+\)', r'\1', text)
    # Headers
    text = re.sub(r'^#{1,6}\s+', '', text, flags=re.MULTILINE)
    # Blockquotes (strip > prefix, keep text)
    text = re.sub(r'^>\s*', '', text, flags=re.MULTILINE)
    # Bold/italic markers
    text = re.sub(r'\*{1,3}([^*]+)\*{1,3}', r'\1', text)
    # Horizontal rules
    text = re.sub(r'^---+\s*$', '', text, flags=re.MULTILINE)
    # HTML tags
    text = re.sub(r'<[^>]+>', '', text)
    return text.strip()


def analyze_text(text):
    """Replicate the slopodar v0.3 analysis engine exactly.
    
    Returns a dict matching the extension's results object.
    """
    results = {
        'totalWords': 0, 'totalSentences': 0,
        'contractions': 0, 'firstPerson': 0, 'questions': 0,
        'transition': 0, 'nomDensity': 0.0,
        'emdash': 0, 'absnoun': 0,
        'epigram': 0, 'isocolon': 0, 'antithesis': 0, 'anadiplosis': 0,
    }
    
    if not text.strip():
        return results
    
    # Word count (line 168 of content.js)
    words = [w for w in text.split() if len(w) > 0]
    results['totalWords'] = len(words)
    
    if results['totalWords'] == 0:
        return results
    
    # Sentence splitting (line 171 of content.js): split(/(?<=[.!?])\s+/)
    sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if s.strip()]
    results['totalSentences'] = len(sentences)
    
    if len(sentences) < 2:
        return results
    
    # Nominalisation density (line 186)
    abs_matches = NOM_RE.findall(text)
    abs_count = len(abs_matches)
    results['absnoun'] = abs_count
    results['nomDensity'] = round((abs_count / results['totalWords']) * 100, 1)
    
    # Contractions (line 194)
    contract_matches = CONTRACT_RE.findall(text)
    results['contractions'] = len(contract_matches)
    
    # First-person (line 199)
    fp_matches = FIRST_PERSON_RE.findall(text)
    results['firstPerson'] = len(fp_matches)
    
    # Questions (line 204)
    results['questions'] = sum(1 for s in sentences if s.strip().endswith('?'))
    
    # Transition words (line 272 — phase 2 highlighter)
    results['transition'] = len(TRANSITION_RE.findall(text))
    
    # Em-dashes (line 309 — count \u2014)
    results['emdash'] = text.count('\u2014')
    
    # Per-sentence analysis (lines 211-261)
    for i, s in enumerate(sentences):
        s_clean = re.sub(r'[.!?]+$', '', s).strip()
        s_words = [w for w in s_clean.split() if len(w) > 0]
        
        # Epigrammatic closure (lines 217-223)
        if i > 0 and 0 < len(s_words) <= 6:
            prev_clean = re.sub(r'[.!?]+$', '', sentences[i-1]).strip()
            prev_words = [w for w in prev_clean.split() if len(w) > 0]
            if len(prev_words) > 10:
                results['epigram'] += 1
        
        # Antithesis (lines 226-237)
        for pattern in ANTI_PATTERNS:
            if pattern.search(s):
                results['antithesis'] += 1
                break
        
        # Pair-based detectors (lines 240-259)
        if i < len(sentences) - 1:
            next_clean = re.sub(r'[.!?]+$', '', sentences[i+1]).strip()
            next_words = [w for w in next_clean.split() if len(w) > 0]
            
            # Isocolon (line 245)
            if (len(s_words) >= 5 and len(next_words) >= 5 and
                    abs(len(s_words) - len(next_words)) <= 1):
                results['isocolon'] += 1
            
            # Anadiplosis (lines 250-259)
            tail = [re.sub(r'[^a-z]', '', w.lower()) for w in s_words[-2:]]
            head = [re.sub(r'[^a-z]', '', w.lower()) for w in next_words[:3]]
            for tw in tail:
                if len(tw) < 4:
                    continue
                if tw in head:
                    results['anadiplosis'] += 1
                    break
    
    return results


def compute_rates(r):
    """Compute per-1k and per-sentence rates from raw counts."""
    tw = r['totalWords']
    ts = r['totalSentences']
    return {
        'totalWords': tw,
        'totalSentences': ts,
        'contractionPer1k': round((r['contractions'] / tw) * 1000, 1) if tw > 0 else 0,
        'firstPersonPer1k': round((r['firstPerson'] / tw) * 1000, 1) if tw > 0 else 0,
        'questionRate': round((r['questions'] / ts) * 100, 1) if ts > 0 else 0,
        'transitionPer1k': round((r['transition'] / tw) * 1000, 1) if tw > 0 else 0,
        'nomDensity': r['nomDensity'],
        'emdashPer1k': round((r['emdash'] / tw) * 1000, 1) if tw > 0 else 0,
        'epigramPer1k': round((r['epigram'] / tw) * 1000, 1) if tw > 0 else 0,
        'isocolonPer1k': round((r['isocolon'] / tw) * 1000, 1) if tw > 0 else 0,
        'antithesisPer1k': round((r['antithesis'] / tw) * 1000, 1) if tw > 0 else 0,
        'anadipPer1k': round((r['anadiplosis'] / tw) * 1000, 1) if tw > 0 else 0,
    }


def sum_score(rates):
    """Composite voice-distance score. Higher = more human-sounding.
    
    sum_score = (contractionPer1k + firstPersonPer1k + questionRate)
              - (transitionPer1k + nomDensity)
    """
    return (rates['contractionPer1k'] + rates['firstPersonPer1k'] + rates['questionRate']) \
         - (rates['transitionPer1k'] + rates['nomDensity'])


print('Analysis engine loaded. Patterns match content.js v0.3.0 exactly.')
print(f'  Contractions: {CONTRACT_RE.pattern[:60]}...')
print(f'  Transitions: {TRANSITION_RE.pattern[:60]}...')
print(f'  First-person: {FIRST_PERSON_RE.pattern}')
print(f'  Nominalisation: {NOM_RE.pattern}')
print(f'  Antithesis patterns: {len(ANTI_PATTERNS)}')

In [ ]:
# Load and score all oceanheart content pages
content_dir = Path('../sites/oceanheart/content')
md_files = sorted(content_dir.rglob('*.md'))

print(f'Found {len(md_files)} .md files in {content_dir}')

oh_rows = []
for md_path in md_files:
    raw = md_path.read_text(encoding='utf-8')
    text = strip_hugo_frontmatter(raw)
    text = strip_markdown(text)
    
    if not text.strip():
        continue
    
    results = analyze_text(text)
    rates = compute_rates(results)
    
    rel_path = str(md_path.relative_to(content_dir))
    oh_rows.append({
        'page': rel_path,
        'section': rel_path.split('/')[0] if '/' in rel_path else 'root',
        **rates,
        'sum_score': sum_score(rates),
    })

oh_df = pd.DataFrame(oh_rows)
print(f'Analysed {len(oh_df)} pages with content')

# Filter to >= 50 words
oh_df = oh_df[oh_df['totalWords'] >= 50].copy()
print(f'After filtering (>=50 words): {len(oh_df)} pages')
print(f'\nSections: {oh_df["section"].value_counts().to_dict()}')

In [ ]:
# Rank all pages by sum_score ascending (worst = most AI-like first)
oh_ranked = oh_df.sort_values('sum_score', ascending=True).reset_index(drop=True)

print('=== Oceanheart Pages Ranked by Voice-Distance Score ===')
print('Lower score = more AI-like. Higher = more human-sounding.\n')
print(f'{"Rank":>4s}  {"Score":>7s}  {"Words":>5s}  {"Contr":>6s}  {"1stP":>6s}  {"Q%":>5s}  {"Trans":>6s}  {"NomD":>5s}  Page')
print('-' * 100)

for i, row in oh_ranked.iterrows():
    print(f'{i+1:>4d}  {row["sum_score"]:>+7.1f}  {row["totalWords"]:>5.0f}  '
          f'{row["contractionPer1k"]:>6.1f}  {row["firstPersonPer1k"]:>6.1f}  '
          f'{row["questionRate"]:>5.1f}  {row["transitionPer1k"]:>6.1f}  '
          f'{row["nomDensity"]:>5.1f}  {row["page"]}')

---
## Section 5: Comparative Analysis

Plot oceanheart page scores against the calibration baselines. Where does the Captain's public writing land?

In [ ]:
# Compute sum_score for calibration data
df['sum_score'] = (df['contractionPer1k'] + df['firstPersonPer1k'] + df['questionRate']) \
                - (df['transitionPer1k'] + df['nomDensity'])

# Score ranges by category
print('=== Sum Score Ranges by Calibration Category ===')
for cat in cat_order:
    subset = df[df['cat'] == cat]
    if len(subset) == 0:
        continue
    print(f'{CAT_LABELS.get(cat, cat):30s}  n={len(subset):>2d}  '
          f'mean={subset["sum_score"].mean():>+7.1f}  '
          f'range=[{subset["sum_score"].min():>+7.1f}, {subset["sum_score"].max():>+7.1f}]')

print(f'\nOceanheart pages (n={len(oh_df)}):'
      f'  mean={oh_df["sum_score"].mean():>+7.1f}  '
      f'  range=[{oh_df["sum_score"].min():>+7.1f}, {oh_df["sum_score"].max():>+7.1f}]')

In [ ]:
# Comparative histogram: calibration categories + oceanheart
fig, ax = plt.subplots(figsize=(14, 6))

# Plot calibration category distributions as violin-style KDEs
for cat in ['A-human-pre', 'C-ai-co', 'F-captain']:
    subset = df[df['cat'] == cat]['sum_score']
    if len(subset) < 2:
        ax.axvline(subset.mean(), color=CAT_COLORS[cat], linestyle='--', linewidth=1.5,
                   label=f'{CAT_LABELS[cat]} (n={len(subset)})', alpha=0.8)
    else:
        try:
            from scipy.stats import gaussian_kde
            kde = gaussian_kde(subset, bw_method=0.4)
            x_range = np.linspace(subset.min() - 10, subset.max() + 10, 200)
            ax.fill_between(x_range, kde(x_range), alpha=0.2, color=CAT_COLORS[cat],
                           label=f'{CAT_LABELS[cat]} (n={len(subset)})')
            ax.plot(x_range, kde(x_range), color=CAT_COLORS[cat], linewidth=1.5)
        except Exception:
            pass

# Scatter oceanheart pages
ax.scatter(oh_df['sum_score'], np.zeros(len(oh_df)) - 0.005,
           color=TOKYO_BLUE, marker='|', s=100, linewidths=1.5,
           label=f'Oceanheart pages (n={len(oh_df)})', zorder=5)

# Category C worst score line
c_worst = df[df['cat'] == 'C-ai-co']['sum_score'].min()
ax.axvline(c_worst, color=TOKYO_RED, linestyle=':', linewidth=1, alpha=0.6)
ax.text(c_worst - 0.5, ax.get_ylim()[1] * 0.85, 'worst C',
        color=TOKYO_RED, fontsize=8, ha='right')

ax.set_xlabel('Sum Score (higher = more human-sounding)')
ax.set_ylabel('Density')
ax.set_title('Oceanheart Pages vs Calibration Baselines')
ax.legend(fontsize=8)
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# How many oceanheart pages fall in each category's range?
print('=== Oceanheart Pages by Calibration Category Range ===')
print()

c_min = df[df['cat'] == 'C-ai-co']['sum_score'].min()
c_max = df[df['cat'] == 'C-ai-co']['sum_score'].max()
a_min = df[df['cat'] == 'A-human-pre']['sum_score'].min()
a_max = df[df['cat'] == 'A-human-pre']['sum_score'].max()

below_c = oh_df[oh_df['sum_score'] < c_min]
in_c_range = oh_df[(oh_df['sum_score'] >= c_min) & (oh_df['sum_score'] <= c_max)]
between = oh_df[(oh_df['sum_score'] > c_max) & (oh_df['sum_score'] < a_min)]
in_a_range = oh_df[(oh_df['sum_score'] >= a_min) & (oh_df['sum_score'] <= a_max)]
above_a = oh_df[oh_df['sum_score'] > a_max]

total = len(oh_df)
print(f'Worse than worst AI blog (< {c_min:+.1f}):  {len(below_c):>3d} / {total} ({100*len(below_c)/total:.0f}%)')
print(f'In AI company range [{c_min:+.1f}, {c_max:+.1f}]:  {len(in_c_range):>3d} / {total} ({100*len(in_c_range)/total:.0f}%)')
print(f'Between AI and human ({c_max:+.1f}, {a_min:+.1f}):  {len(between):>3d} / {total} ({100*len(between)/total:.0f}%)')
print(f'In human range [{a_min:+.1f}, {a_max:+.1f}]:  {len(in_a_range):>3d} / {total} ({100*len(in_a_range)/total:.0f}%)')
print(f'Above best human (> {a_max:+.1f}):  {len(above_a):>3d} / {total} ({100*len(above_a)/total:.0f}%)')

if len(below_c) > 0:
    print(f'\n--- Pages scoring worse than worst AI blog ---')
    for _, row in below_c.sort_values('sum_score').iterrows():
        print(f'  {row["sum_score"]:>+7.1f}  {row["page"]}')

In [ ]:
# Captain's blog posts vs Captain's calibration samples (Category F)
cat_f = df[df['cat'] == 'F-captain']

print('=== Captain Calibration Samples (Category F) ===')
for _, row in cat_f.iterrows():
    print(f'  {row["label"]:30s}  score={row["sum_score"]:>+7.1f}  '
          f'contr={row["contractionPer1k"]:>5.1f}  1stP={row["firstPersonPer1k"]:>5.1f}  '
          f'Q%={row["questionRate"]:>5.1f}  trans={row["transitionPer1k"]:>5.1f}  '
          f'nom={row["nomDensity"]:>5.1f}')

f_mean = cat_f['sum_score'].mean()
f_min = cat_f['sum_score'].min()
f_max = cat_f['sum_score'].max()
print(f'\nCaptain calibration range: [{f_min:+.1f}, {f_max:+.1f}], mean={f_mean:+.1f}')

# Oceanheart blog posts specifically
oh_blog = oh_df[oh_df['section'] == 'blog'].copy()
if len(oh_blog) > 0:
    print(f'\n=== Oceanheart Blog Posts vs Captain Calibration ===')
    for _, row in oh_blog.sort_values('sum_score').iterrows():
        in_f = 'IN F' if f_min <= row['sum_score'] <= f_max else 'OUT'
        print(f'  {row["sum_score"]:>+7.1f}  [{in_f:>4s}]  {row["page"]}')
    
    oh_in_f = oh_blog[(oh_blog['sum_score'] >= f_min) & (oh_blog['sum_score'] <= f_max)]
    print(f'\n{len(oh_in_f)} / {len(oh_blog)} blog posts fall within Captain\'s calibration range')
else:
    print('\nNo blog posts found with >= 50 words.')

---
## Section 6: Enhancement — What the JS Couldn't Do

Python + scipy + statsmodels + sklearn make several analyses trivial that would be hard in vanilla JS.

In [ ]:
# ── 6.1: Principal Component Analysis ──
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

feature_cols = ['contractionPer1k', 'firstPersonPer1k', 'questionRate',
                'transitionPer1k', 'nomDensity', 'hedgePer1k',
                'emdashPer1k', 'epigramPer1k', 'isocolonPer1k',
                'antithesisPer1k', 'anadipPer1k']

X = df[feature_cols].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA()
X_pca = pca.fit_transform(X_scaled)

print('=== PCA: Variance Explained ===')
cumvar = np.cumsum(pca.explained_variance_ratio_)
for i, (var, cum) in enumerate(zip(pca.explained_variance_ratio_, cumvar)):
    print(f'  PC{i+1}: {var:.3f} ({cum:.3f} cumulative)')
    if cum > 0.95:
        break

print(f'\nPC1 + PC2 explain {cumvar[1]:.1%} of variance.')

In [ ]:
# PCA biplot: PC1 vs PC2, colored by category
fig, ax = plt.subplots(figsize=(12, 8))

for cat in cat_order:
    mask = df['cat'] == cat
    if mask.sum() == 0:
        continue
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
               c=CAT_COLORS[cat], label=CAT_LABELS[cat],
               s=60, alpha=0.8, edgecolors='white', linewidth=0.5)

# Loading vectors
loadings = pca.components_[:2].T
scale = 3  # scale for visibility
for i, feat in enumerate(feature_cols):
    ax.annotate('', xy=(loadings[i, 0] * scale, loadings[i, 1] * scale),
                xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color=TOKYO_COMMENT, lw=1))
    # Offset labels to avoid overlap
    offset = 0.15
    ax.text(loadings[i, 0] * scale + offset, loadings[i, 1] * scale + offset,
            feat.replace('Per1k', '').replace('Rate', '%'),
            fontsize=7, color=TOKYO_FG, alpha=0.8)

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
ax.set_title('PCA Biplot: Feature Space Decomposition')
ax.legend(fontsize=8, loc='best')
ax.grid(alpha=0.2)
ax.axhline(0, color=TOKYO_COMMENT, linewidth=0.5)
ax.axvline(0, color=TOKYO_COMMENT, linewidth=0.5)

plt.tight_layout()
plt.show()

In [ ]:
# ── 6.2: Hierarchical Clustering ──
from scipy.cluster.hierarchy import dendrogram, linkage

Z = linkage(X_scaled, method='ward')

fig, ax = plt.subplots(figsize=(16, 6))

# Color labels by category
label_colors = {cat: CAT_COLORS[cat] for cat in cat_order}
labels = df['label'].values
cats = df['cat'].values

dendro = dendrogram(Z, labels=labels, ax=ax, leaf_rotation=90, leaf_font_size=7,
                    color_threshold=0)

# Color the x-axis labels by category
xlabels = ax.get_xticklabels()
for lbl in xlabels:
    text = lbl.get_text()
    # Find the category for this label
    mask = df['label'] == text
    if mask.any():
        cat = df.loc[mask, 'cat'].iloc[0]
        lbl.set_color(CAT_COLORS.get(cat, TOKYO_FG))

ax.set_title('Hierarchical Clustering (Ward\'s method) — Do Categories Emerge from Features?')
ax.set_ylabel('Ward distance')

# Manual legend
from matplotlib.patches import Patch
legend_handles = [Patch(facecolor=CAT_COLORS[c], label=CAT_LABELS[c])
                  for c in cat_order if c in df['cat'].values]
ax.legend(handles=legend_handles, fontsize=7, loc='upper right')

plt.tight_layout()
plt.show()

In [ ]:
# ── 6.3: Logistic Regression ──
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.metrics import classification_report, roc_curve, auc

# Binary classification: A (human) vs C (AI)
ac_mask = df['cat'].isin(['A-human-pre', 'C-ai-co'])
X_ac = scaler.fit_transform(df.loc[ac_mask, feature_cols].values)
y_ac = (df.loc[ac_mask, 'cat'] == 'A-human-pre').astype(int).values  # 1 = human, 0 = AI

print(f'Binary classification: A (human, n={y_ac.sum()}) vs C (AI, n={len(y_ac) - y_ac.sum()})\n')

# Leave-one-out cross-validation (appropriate for small n)
lr = LogisticRegression(max_iter=1000, random_state=42)
loo = LeaveOneOut()

y_pred = cross_val_predict(lr, X_ac, y_ac, cv=loo)
y_prob = cross_val_predict(lr, X_ac, y_ac, cv=loo, method='predict_proba')[:, 1]

print('=== Leave-One-Out Cross-Validation Results ===')
print(classification_report(y_ac, y_pred, target_names=['AI (C)', 'Human (A)']))

accuracy = (y_pred == y_ac).mean()
print(f'Overall LOO accuracy: {accuracy:.1%}')

# Feature importance from full model
lr.fit(X_ac, y_ac)
print('\n=== Logistic Regression Coefficients ===')
coefs = sorted(zip(feature_cols, lr.coef_[0]), key=lambda x: abs(x[1]), reverse=True)
for feat, coef in coefs:
    direction = 'human+' if coef > 0 else 'AI+'
    print(f'  {feat:25s}  {coef:>+7.3f}  ({direction})')

In [ ]:
# ── 6.4: ROC Curves for Top Discriminators ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: full model ROC
ax = axes[0]
fpr, tpr, _ = roc_curve(y_ac, y_prob)
roc_auc = auc(fpr, tpr)
ax.plot(fpr, tpr, color=TOKYO_BLUE, linewidth=2,
        label=f'Full model (AUC = {roc_auc:.2f})')
ax.plot([0, 1], [0, 1], color=TOKYO_COMMENT, linestyle='--', linewidth=0.8)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC: Full Model (LOO CV)')
ax.legend(fontsize=9)
ax.grid(alpha=0.2)

# Right: individual feature ROCs
ax = axes[1]
top_features = ['transitionPer1k', 'nomDensity', 'questionRate',
                'firstPersonPer1k', 'contractionPer1k']
feature_colors = [TOKYO_RED, TOKYO_ORANGE, TOKYO_GREEN, TOKYO_CYAN, TOKYO_PURPLE]

for feat, color in zip(top_features, feature_colors):
    vals = df.loc[ac_mask, feat].values
    # For AI-higher features, flip sign so higher = more AI
    if feat in ['transitionPer1k', 'nomDensity']:
        fpr_f, tpr_f, _ = roc_curve(y_ac, -vals)
    else:
        fpr_f, tpr_f, _ = roc_curve(y_ac, vals)
    roc_auc_f = auc(fpr_f, tpr_f)
    short = feat.replace('Per1k', '').replace('Rate', '%')
    ax.plot(fpr_f, tpr_f, color=color, linewidth=1.5,
            label=f'{short} (AUC={roc_auc_f:.2f})')

ax.plot([0, 1], [0, 1], color=TOKYO_COMMENT, linestyle='--', linewidth=0.8)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC: Individual Top Features')
ax.legend(fontsize=8)
ax.grid(alpha=0.2)

plt.tight_layout()
plt.show()

In [ ]:
# ── 6.5: Correlation Heatmap ──
fig, ax = plt.subplots(figsize=(10, 8))

corr = df[feature_cols].corr()

# Custom colormap: Tokyo Night
from matplotlib.colors import LinearSegmentedColormap
tokyo_cmap = LinearSegmentedColormap.from_list('tokyo',
    [TOKYO_RED, TOKYO_BG, TOKYO_BLUE])

sns.heatmap(corr, annot=True, fmt='.2f', cmap=tokyo_cmap,
            center=0, vmin=-1, vmax=1,
            square=True, ax=ax,
            xticklabels=[f.replace('Per1k', '').replace('Rate', '%') for f in feature_cols],
            yticklabels=[f.replace('Per1k', '').replace('Rate', '%') for f in feature_cols],
            annot_kws={'size': 7},
            linewidths=0.5, linecolor=TOKYO_GRID)

ax.set_title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# ── 6.6: Score the Oceanheart Pages on the Logistic Model ──
# Apply the fitted logistic regression to predict P(human) for each oceanheart page
# Note: hedgePer1k exists in the calibration TSV but isn't computed by the slopodar
# engine (it's not one of the extension's heuristics). Re-fit on shared features.

oh_feature_cols = [c for c in feature_cols if c in oh_df.columns]
missing_feats = [c for c in feature_cols if c not in oh_df.columns]
if missing_feats:
    print(f'Features in calibration but not computed for oceanheart: {missing_feats}')
    print(f'Re-fitting logistic regression on {len(oh_feature_cols)} shared features.\n')
    from sklearn.preprocessing import StandardScaler as SS
    scaler_shared = SS()
    X_ac_shared = scaler_shared.fit_transform(df.loc[ac_mask, oh_feature_cols].values)
    lr_shared = LogisticRegression(max_iter=1000, random_state=42)
    lr_shared.fit(X_ac_shared, y_ac)
else:
    lr_shared = lr
    scaler_shared = scaler

oh_features = oh_df[oh_feature_cols].values
oh_scaled = scaler_shared.transform(oh_features)
oh_df['p_human'] = lr_shared.predict_proba(oh_scaled)[:, 1]
oh_df['predicted'] = ['Human' if p > 0.5 else 'AI-like' for p in oh_df['p_human']]

print('=== Oceanheart Pages: Logistic Regression P(human) ===')
print('Model trained on A (human) vs C (AI). P(human) > 0.5 = classified human.\n')

for _, row in oh_df.sort_values('p_human').iterrows():
    marker = '  ' if row['p_human'] > 0.5 else '>>' 
    print(f'{marker} P(human)={row["p_human"]:.3f}  score={row["sum_score"]:>+7.1f}  '
          f'{row["totalWords"]:>4.0f}w  {row["page"]}')

n_ai_like = (oh_df['p_human'] <= 0.5).sum()
print(f'\n{n_ai_like} / {len(oh_df)} pages classified as AI-like by the logistic model')
print(f'\nCAVEAT: This model was trained on {y_ac.sum()} human + {len(y_ac)-y_ac.sum()} AI samples.')
print('It knows what "tech blog human" and "AI company blog" look like.')
print('It does NOT know what "human writing about AI" looks like — a category not in training.')

---
## Section 7: Summary & Recommendations

### Key Findings

In [ ]:
# Automated summary
print('=' * 70)
print('SUMMARY: Slopodar Calibration Analysis')
print('=' * 70)

print('\n1. TOP DISCRIMINATORS (A vs C, ranked by |Cohen\'s d|):')
print()
top5_features = es_df[es_df['feature'].isin(voice_cols)].head(5)
for rank, (_, row) in enumerate(top5_features.iterrows(), 1):
    # Find corresponding MW result
    mw_match = mw_df[mw_df['feature'] == row['feature']]
    p_str = f'p={mw_match.iloc[0]["p_bonferroni"]:.4f}' if len(mw_match) > 0 else 'p=n/a'
    d_str = f'd={row["cohens_d"]:+.2f}' if not np.isinf(row['cohens_d']) else 'd=inf'
    print(f'   #{rank}: {row["feature"]:25s}  {d_str}  {p_str} (Bonferroni)  {row["direction"]}')

print(f'\n2. FORMAL TESTS:')
print(f'   Mann-Whitney U significant after Bonferroni: {mw_df["significant"].sum()} / {len(mw_df)} features')
print(f'   Logistic regression LOO accuracy: {accuracy:.1%}')
print(f'   Full model AUC: {roc_auc:.2f}')

print(f'\n3. OCEANHEART STATE:')
print(f'   Pages analysed (>=50 words): {len(oh_df)}')
print(f'   Mean sum_score: {oh_df["sum_score"].mean():+.1f}')
cat_a_mean = df[df['cat'] == 'A-human-pre']['sum_score'].mean()
cat_c_mean = df[df['cat'] == 'C-ai-co']['sum_score'].mean()
print(f'   Category A mean: {cat_a_mean:+.1f}, Category C mean: {cat_c_mean:+.1f}')
print(f'   Pages in AI range: {len(in_c_range)} ({100*len(in_c_range)/total:.0f}%)')
print(f'   Pages worse than worst AI: {len(below_c)} ({100*len(below_c)/total:.0f}%)')
print(f'   Pages in human range: {len(in_a_range)} ({100*len(in_a_range)/total:.0f}%)')
n_ai_like_lr = (oh_df['p_human'] <= 0.5).sum()
print(f'   Classified AI-like by logistic model: {n_ai_like_lr} ({100*n_ai_like_lr/len(oh_df):.0f}%)')

print(f'\n4. PCA INSIGHT:')
print(f'   PC1+PC2 explain {cumvar[1]:.1%} of feature variance.')
print(f'   The primary axis separates voice presence (contractions, first-person, questions)')
print(f'   from formal register (transitions, nominalisation density).')
print(f'   This IS the voice-distance dimension. The slopodar sum_score is a')
print(f'   reasonable approximation of PC1.')

print(f'\n5. WHAT WOULD MOVE THE NEEDLE:')
print(f'   a) Add contractions. Any contraction presence is a strong human signal.')
print(f'      Current oceanheart contraction rate: {oh_df["contractionPer1k"].mean():.1f}/k')
print(f'      Category A mean: {cat_a["contractionPer1k"].mean():.1f}/k')
print(f'   b) Add questions. Rhetorical questions signal a mind thinking aloud.')
print(f'      Current oceanheart question rate: {oh_df["questionRate"].mean():.1f}%')
print(f'      Category A mean: {cat_a["questionRate"].mean():.1f}%')
print(f'   c) Increase first-person pronouns. Presence in the text.')
print(f'      Current oceanheart: {oh_df["firstPersonPer1k"].mean():.1f}/k')
print(f'      Category A mean: {cat_a["firstPersonPer1k"].mean():.1f}%')
print(f'   d) Remove transition words. "However", "Moreover", "Furthermore" are tells.')
print(f'      Current oceanheart: {oh_df["transitionPer1k"].mean():.1f}/k')
print(f'      Category A mean: {cat_a["transitionPer1k"].mean():.1f}/k')

### Interpretation Notes

**What the numbers say:** The calibration data, despite its small size, shows a clear and statistically robust separation between pre-LLM human writing and AI company blogs along a voice-distance dimension defined primarily by five features: transition word density, nominalisation density, question rate, first-person pronoun rate, and contraction rate. These findings survive Bonferroni correction and hold up under non-parametric testing.

**What the numbers don't say:** The calibration corpus is 47 entries from a narrow demographic slice. The "human" baseline is mostly male American tech bloggers. The "AI" baseline is corporate AI company blogs — a specific genre with its own conventions independent of whether an LLM wrote them. A human writing a formal corporate blog would score similarly to Category C. A carefully prompted LLM writing in PG's style might score like Category A. The tool measures *voice distance from a particular human register*, not *authorship*.

**The oceanheart diagnosis:** The blog posts on oceanheart.ai show the expected pattern for agent-drafted content that a human has reviewed but not rewritten: lower-than-human contraction rates, lower question rates, and higher transition word density. The draft notice on these pages is honest. The tool confirms what the notice says.

**The confounds, stated plainly:**
- **Construct Drift (#13):** The thing we're measuring ("LLM-like writing") is a moving target. As LLMs are fine-tuned on human feedback and as humans adapt their writing to AI tools, the distance between the distributions will change.
- **Demographic Bake-In (#14):** The human baseline is not "human writing" — it is "male American tech blogger writing from 2000-2020." Every claim is bounded by this provenance.
- **Monoculture Analysis (#15):** Using features derived from one genre to score another genre (session decisions, CVs, research pages) is a category error. The research pages should be scored against a different baseline.

These are not caveats appended for politeness. They are the actual boundaries of what this analysis can claim.